In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms

In [17]:
transform = transforms.ToTensor()

In [18]:
train_dataset = torchvision.datasets.FashionMNIST(root='./data',train=True,download=True,transform=transform)

In [19]:
test_dataset = torchvision.datasets.FashionMNIST(root='./data',train=False,download=True,transform=transform)

In [20]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [28]:
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [29]:
class FashionCNN(nn.Module):
    def __init__(self):
        super(FashionCNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [31]:
model = FashionCNN().to(device)

In [32]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
for epoch in range(5):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    print(f"Epoch {epoch+1} Loss: {running_loss/len(train_loader):.4f}")

Epoch 1 Loss: 0.0511


In [ ]:
model.eval()
correct = 0
total = 0

In [ ]:
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy = {100 * correct / total:.2f}%")